In [ ]:
import google.generativeai as genai
import pandas as pd
import json
import time
from datetime import datetime

# Configure your Gemini API
from google.colab import userdata
API_KEY=userdata.get('GEMINI_API_KEY')
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('gemini-1.5-flash-latest')

def extract_emails(domain):
    """Extract emails from a domain"""

    prompt = f"""
    Analyze the website https://{domain} and find:
    1. Any email addresses from this domain
    2. If no emails found, find the parent company's email

    Return ONLY this JSON format:
    {{
        "domain": "{domain}",
        "primary_email": "found_email@{domain}",
        "parent_email": "parent_email@parentcompany.com",
        "status": "success"
    }}

    If no emails found at all, set emails to empty string "".
    If website doesn't work, set status to "failed".
    """

    try:
        response = model.generate_content(prompt)
        result_text = response.text.strip()

        # Clean the response
        if '```json' in result_text:
            result_text = result_text.split('```json')[1].split('```')[0]
        elif '```' in result_text:
            result_text = result_text.split('```')[1]

        result = json.loads(result_text.strip())
        return result

    except Exception as e:
        print(f"Error with {domain}: {e}")
        return {
            "domain": domain,
            "primary_email": "",
            "parent_email": "",
            "status": "failed"
        }



Starting email extraction...
Processing 3 domains...
Processing 1/3: xtool.eu
Processing 2/3: tryglowora.eu
Processing 3/3: taketinyshield.com

Saved to: emails_20250911_1130.xlsx

SUMMARY:
Total domains: 3
Found emails: 0
Success rate: 0.0%


In [ ]:
def process_domains(domain_list):
    """Process multiple domains"""
    results = []

    for i, domain in enumerate(domain_list, 1):
        print(f"Processing {i}/{len(domain_list)}: {domain}")

        # Clean domain name
        domain = domain.replace('https://', '').replace('http://', '').replace('www.', '').strip('/')

        result = extract_emails(domain)
        results.append(result)

        # Wait 2 seconds between requests
        if i < len(domain_list):
            time.sleep(2)

    return results

In [ ]:
def save_to_excel(results):
    """Save results to Excel file"""

    # Create simple data for Excel
    data = []
    for result in results:
        data.append({
            'Domain': result['domain'],
            'Primary Email': result['primary_email'],
            'Parent Email': result['parent_email'],
            'Status': result['status'],
            'Date': datetime.now().strftime('%Y-%m-%d')
        })

    # Save to Excel
    df = pd.DataFrame(data)
    filename = f'emails_{datetime.now().strftime("%Y%m%d_%H%M")}.xlsx'
    df.to_excel(filename, index=False)

    print(f"\nSaved to: {filename}")
    return filename

In [ ]:
domains = [
        "xtool.eu",
        "tryglowora.eu",
        "taketinyshield.com"
    ]
print("Starting email extraction...")
print(f"Processing {len(domains)} domains...")

# Extract emails
results = process_domains(domains)

# Save to Excel
save_to_excel(results)

# Show summary
successful = len([r for r in results if r['primary_email'] or r['parent_email']])
print(f"\nSUMMARY:")
print(f"Total domains: {len(domains)}")
print(f"Found emails: {successful}")
print(f"Success rate: {successful/len(domains)*100:.1f}%")